## Project-02 & 03: Matryoshka Method & Benchmarking Metrics

## 1. Matryoshka Representation Learning (MRL)
***Matryoshka*** Representation Learning is an advanced embedding technique inspired by Russian nesting dolls. Traditional embedding models distribute semantic information uniformly across all dimensions, meaning that if a vector is truncated, it loses its meaning entirely. In contrast, MRL forces the deep learning model to concentrate the most critical semantic features into the initial dimensions of the vector. This allows developers to truncate (slice) high-dimensional vectors (e.g., from 1536 down to 256 or 64 dimensions) while retaining the vast majority of their original semantic integrity. 

This approach significantly reduces memory footprint (RAM/Storage) and accelerates distance calculations (Latency) with a negligible drop in retrieval accuracy.

**Supported Models for Matryoshka Representation Learning**
Today, in addition to proprietary API-based models like OpenAI's `text-embedding-3-small` and `text-embedding-3-large`, the open-source community has rapidly adopted this architecture. Several powerful open-source models available on the Hugging Face platform natively support Matryoshka Representation Learning (MRL). Prominent examples include:
* `nomic-ai/nomic-embed-text-v1.5`: A highly efficient model designed specifically with Matryoshka capabilities, allowing dynamic dimension resizing without retraining.
* `BAAI/bge-m3`: A  multilingual model that supports dense retrieval and varying embedding sizes.

## 2. Search Benchmarking Metrics in Vector Retrieval
> **Recall@K:** Recall@K measures the proportion of relevant documents successfully retrieved within the top K results. It answers the fundamental question: "Is the correct answer present in the first K options?" It is a binary metric per query (1 if found, 0 if not). While it ensures the document was found, it does not evaluate where it was placed within those K results.

> **mAP (Mean Average Precision):** mAP evaluates not just the retrieval of relevant documents, but their ranking quality. Average Precision (AP) rewards systems that place the correct answer higher in the result list (e.g., rank 1 is much better than rank 10). mAP is simply the mean of the AP scores across all test queries. It is calculated as   $1 / Rank$.

> **NDCG (Normalized Discounted Cumulative Gain):** NDCG is an industry-standard metric for search engines that evaluates the ranking of retrieved items by applying a logarithmic discount. The lower a relevant document appears in the search results, the heavier the penalty it receives. NDCG is particularly powerful because it can handle multi-graded relevance (where some documents are "highly relevant" and others are only "partially relevant"). For a single ground-truth target, the formula simplifies to   $1 / log2(Rank + 1)$.

In [10]:
import os
import time
import math
import random
import itertools
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
# 1. Setup & Configuration
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip().replace('"', '').replace("'", "")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.avalai.ir/v1").strip()

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embeddings_batch(texts: list) -> np.ndarray:
    """Fetch embeddings for multiple texts in a single API call (Batch Processing)."""
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    # Sort the response by index to maintain the original order
    sorted_data = sorted(response.data, key=lambda x: x.index)
    return np.array([data.embedding for data in sorted_data])

In [4]:
# 2. Dataset & Multi-Query Ground Truth
dataset_texts = [
    "Ali is 16 years old and lives in Karaj.",
    "Python is a high-level programming language.",
    "The Eiffel Tower is located in Paris, France.",
    "Machine learning is a subset of artificial intelligence.",
    "Water boils at 100 degrees Celsius at standard pressure.",
    "The capital of Japan is Tokyo.",
    "JavaScript is primarily used for web development.",
    "Mount Everest is the highest mountain in the world.",
    "Photosynthesis is how plants convert light into energy.",
    "Albert Einstein developed the theory of relativity.",
    "The speed of light is approximately 299,792 kilometers per second.",
    "Shakespeare wrote Romeo and Juliet.",
    "A vector database stores high-dimensional vector representations.",
    "SQL databases use tables to store relational data.",
    "The human body has 206 bones in adulthood.",
    "The Great Wall of China is visible from space.",
    "Ollama allows you to run large language models locally.",
    "Pinecone is a managed, cloud-native vector database.",
    "RAG stands for Retrieval-Augmented Generation.",
    "React is a popular JavaScript library for building user interfaces.",
    "The mitochondria is the powerhouse of the cell.",
    "A standard guitar usually has 6 strings.",
    "The Pacific Ocean is the largest ocean on Earth.",
    "Leonardo da Vinci painted the Mona Lisa.",
    "Atoms are composed of protons, neutrons, and electrons.",
    "The freezing point of water is 0 degrees Celsius.",
    "HTML stands for HyperText Markup Language.",
    "Chess is a two-player strategy board game.",
    "Mars is often called the Red Planet.",
    "Gravity keeps the Earth in orbit around the Sun."
]

print("⏳ Fetching 1536-dimensional embeddings in batches...")
dataset_embeddings = get_embeddings_batch(dataset_texts)

# Defining multiple test cases: (Query String, Expected Target Index)
test_cases = [
    ("What city is the capital of Japan?", 5),
    ("What is the powerhouse of the cell?", 20),
    ("Who is the author of Romeo and Juliet?", 11)
]

queries = [tc[0] for tc in test_cases]
query_embeddings_full = get_embeddings_batch(queries)

⏳ Fetching 1536-dimensional embeddings in batches...


In [5]:
# 3. Matryoshka Truncation & Benchmarking
dimensions_to_test = [1536, 512, 128, 64]
results_list = []

print("\n Starting Advanced Matryoshka Benchmark...")

for dim in dimensions_to_test:
    # 1. Truncate Database and Queries
    trunc_dataset = dataset_embeddings[:, :dim]
    trunc_queries = query_embeddings_full[:, :dim]
    
    # 2. Re-normalize vectors (Crucial for Matryoshka cosine similarity)
    trunc_dataset = trunc_dataset / np.linalg.norm(trunc_dataset, axis=1, keepdims=True)
    trunc_queries = trunc_queries / np.linalg.norm(trunc_queries, axis=1, keepdims=True)
    
    memory_bytes = trunc_dataset.nbytes
    
    # 3. Measure Latency (Simulating 1000 batch searches for stable average)
    start_time = time.perf_counter()
    for _ in range(1000):
        _ = cosine_similarity(trunc_queries, trunc_dataset)
    end_time = time.perf_counter()
    
    avg_latency_us = ((end_time - start_time) / 1000) * 1_000_000 
    
    # 4. Calculate Precision Metrics (Recall, mAP, NDCG)
    recalls = []
    aps = []
    ndcgs = []
    
    # Calculate similarity for all queries at once
    sims = cosine_similarity(trunc_queries, trunc_dataset)
    
    for i, (query, expected_index) in enumerate(test_cases):
        # Get top 10 indices
        top_10_indices = np.argsort(sims[i])[::-1][:10]
        
        if expected_index in top_10_indices:
            recalls.append(1.0)
            
            # Find the rank (1-based index)
            rank = np.where(top_10_indices == expected_index)[0][0] + 1
            
            # AP formula for single relevant document: 1 / rank
            aps.append(1.0 / rank)
            
            # NDCG formula for single relevant document: 1 / log2(rank + 1)
            ndcgs.append(1.0 / math.log2(rank + 1))
        else:
            recalls.append(0.0)
            aps.append(0.0)
            ndcgs.append(0.0)
            
    # Average the metrics across all test queries
    avg_recall = np.mean(recalls)
    avg_map = np.mean(aps)
    avg_ndcg = np.mean(ndcgs)
    
    results_list.append({
        'Dimension': dim,
        'Memory (Bytes)': memory_bytes,
        'Latency (µs)': avg_latency_us,
        'Recall@10': avg_recall,
        'mAP@10': avg_map,
        'NDCG@10': avg_ndcg
    })

print("✅ Advanced Benchmark Completed!\n")

df_results = pd.DataFrame(results_list)
df_results.set_index('Dimension', inplace=True)

styled_df = (df_results.style
             .background_gradient(cmap='Greens_r', subset=['Memory (Bytes)', 'Latency (µs)'])
             .background_gradient(cmap='Greens', subset=['Recall@10', 'mAP@10', 'NDCG@10'])
             .format({
                 'Memory (Bytes)': '{:,.0f}',
                 'Latency (µs)': '{:.2f}',
                 'Recall@10': '{:.2f}',
                 'mAP@10': '{:.2f}',
                 'NDCG@10': '{:.2f}'
             }))

styled_df


 Starting Advanced Matryoshka Benchmark...
✅ Advanced Benchmark Completed!



,Memory (Bytes),Latency (µs),Recall@10,mAP@10,NDCG@10
Dimension,,,,,
1536,"368,640",463.65,1.00,1.00,1.00
512,"122,880",284.96,1.00,1.00,1.00
128,"30,720",243.11,1.00,1.00,1.00
64,"15,360",237.26,1.00,1.00,1.00


---

## ***initial Benchmark Results Analysis (Tasks 2 & 3)***

The results of the dimensionality reduction experiment using the `text-embedding-3-small` model demonstrate the following key insights:

#### 1. Memory Footprint Analysis (Linear Reduction)
As shown in the table, memory consumption decreases linearly with dimensionality.
* At 1536 dimensions, the dataset matrix requires 368,640 bytes.
* When truncated to 64 dimensions, this drops to just 15,360 bytes.
* **Conclusion:** We achieved an impressive **24x reduction (~95.8%)** in RAM and storage requirements, which translates to massive savings in large-scale production infrastructure.

#### 2. Search Latency Analysis (Speed Improvement)
The time required to compute the cosine similarity (matrix dot product) decreased significantly.
* **Conclusion:** Dimensionality reduction resulted in an approximate **40% increase in search speed**. Implementing this in a dedicated C++ vector engine (like Pinecone/ChromaDB) would yield even more substantial latency improvements.

#### 3. Accuracy Metrics & The "Dataset Variance" Caveat
In this specific baseline test, despite discarding nearly 96% of the vector data (from 1536 to 64 dims), our retrieval accuracy metrics (Recall@10, mAP@10, NDCG@10) remained at **1.00 (100%)**. 

* **Why didn't accuracy drop here?** This perfect score is a combined result of Matryoshka Representation Learning (MRL)—which front-loads critical semantic features—and our specific dataset configuration. Our baseline dataset contained 30 highly distinct and semantically distant facts (e.g., Biology vs. Geography). Differentiating between completely unrelated topics requires very few dimensions.
* **Important Caveat (Stress Condition):** While 64 dimensions are sufficient for a small, high-variance dataset, deploying this in a dense production environment (e.g., millions of records or highly overlapping semantic concepts) will expose the limitations of extreme truncation. To prove this, a **Stress Test** with a much larger, semantically dense dataset is required to observe the accuracy degradation at lower dimensions.

---

### Stress Test Dataset Generation (1000 times)

In [ ]:
def get_embeddings_in_chunks(texts: list, chunk_size=100) -> np.ndarray:
    """Fetch embeddings in safe chunks to avoid API Payload limits or Timeouts."""
    all_embeddings = []
    print(f"   -> Fetching {len(texts)} embeddings in chunks of {chunk_size}...")
    for i in range(0, len(texts), chunk_size):
        chunk = texts[i:i + chunk_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=chunk)
        # Sort by index to ensure order is maintained within the chunk
        sorted_data = sorted(response.data, key=lambda x: x.index)
        all_embeddings.extend([data.embedding for data in sorted_data])
        print(f"      [+] Processed {min(i + chunk_size, len(texts))}/{len(texts)}")
        time.sleep(0.5) # Short pause to respect API rate limits
    return np.array(all_embeddings)

print("⏳ Generating 1000 diverse and highly dense sentences for STRESS TEST...")
random.seed(42) # For reproducibility

# Group A: 600 Highly Similar Sentences (Medium Length)
names = ["Ali", "Sara", "Reza", "Mina", "Hassan", "Zahra", "Omid", "Neda", "Sina", "Maryam"] # 10
cities = ["Tehran", "Karaj", "Isfahan", "Shiraz", "Mashhad", "Tabriz"] # 6
skills = ["Python", "Java", "C++", "JavaScript", "Go", "Rust", "Ruby", "Swift", "Kotlin", "PHP"] # 10
# 10 * 6 * 10 = 600 combinations
tech_sentences = [f"The software engineer named {n} is currently residing in {c} and specializes in {s} development." for n, c, s in itertools.product(names, cities, skills)]

# Group B: 200 Short Random Facts (Noise)
subjects = ["Water", "Fire", "Earth", "Air", "Space", "Time", "Gravity", "Light"]
actions = ["moves quickly", "stands still", "heats up", "cools down", "expands"]
objects = ["in the morning.", "at night.", "during winter.", "in a vacuum.", "under pressure."]
# 8 * 5 * 5 = 200 combinations
short_sentences = [f"{sub} {act} {obj}" for sub, act, obj in itertools.product(subjects, actions, objects)]

# Group C: 200 Long Complex Paragraphs (Semantic Distraction)
themes = ["Artificial Intelligence", "Climate Change", "Space Exploration", "Quantum Mechanics", "Ancient History"]
impacts = ["is transforming how we live and work daily,", "poses significant challenges to modern society,", "opens up entirely new frontiers of discovery,"]
details = ["requiring global cooperation and massive investment to ensure a sustainable future for all.", "often leading to unexpected philosophical and ethical debates among top scholars.", "pushing the boundaries of what humanity once thought was scientifically possible.", "which demands rigorous testing, validation, and peer review before implementation."]
long_sentences = []
for i in range(200):
    t = random.choice(themes)
    imp = random.choice(impacts)
    d = random.choice(details)
    long_sentences.append(f"The extensive study of {t} {imp} {d} This has been documented in numerous peer-reviewed research papers and academic journals (Reference ID: {i}).")

# Combine all groups to create exactly 1000 items
dataset_texts = tech_sentences + short_sentences + long_sentences

print("⏳ Calling OpenAI/AvalAI API to generate 1000 embeddings...")
dataset_embeddings = get_embeddings_in_chunks(dataset_texts, chunk_size=200)

# Helper function to find the exact index of our target answer in Group A
def get_target_index(n, c, s):
    target = f"The software engineer named {n} is currently residing in {c} and specializes in {s} development."
    return dataset_texts.index(target)

# Define complex queries that test subtle differences
test_cases = [
    ("Looking for a Go developer based in Isfahan named Reza.", get_target_index("Reza", "Isfahan", "Go")),
    ("I need a Rust programmer from Mashhad, ideally her name is Mina.", get_target_index("Mina", "Mashhad", "Rust")),
    ("Where does Hassan live and what does he code? He writes Java in Shiraz.", get_target_index("Hassan", "Shiraz", "Java")),
    ("Find the profile for Ali. He is a C++ expert located in Tehran.", get_target_index("Ali", "Tehran", "C++"))
]

queries = [tc[0] for tc in test_cases]
query_embeddings_full = get_embeddings_in_chunks(queries, chunk_size=10)

# 3. Matryoshka Benchmarking
dimensions_to_test = [1536, 512, 128, 64, 32] # Added 32 dimensions to really push the model to its breaking point
results_list = []

print("\n Starting 1000-Item STRESS TEST Benchmark...")

for dim in dimensions_to_test:
    trunc_dataset = dataset_embeddings[:, :dim]
    trunc_queries = query_embeddings_full[:, :dim]
    
    trunc_dataset = trunc_dataset / np.linalg.norm(trunc_dataset, axis=1, keepdims=True)
    trunc_queries = trunc_queries / np.linalg.norm(trunc_queries, axis=1, keepdims=True)
    
    memory_bytes = trunc_dataset.nbytes
    
    start_time = time.perf_counter()
    for _ in range(100): # 100 iterations for stable latency testing
        _ = cosine_similarity(trunc_queries, trunc_dataset)
    end_time = time.perf_counter()
    
    avg_latency_us = ((end_time - start_time) / 100) * 1_000_000 
    
    recalls = []
    aps = []
    ndcgs = []
    
    sims = cosine_similarity(trunc_queries, trunc_dataset)
    
    # Use Top 5 (Recall@5) to make it highly strict!
    K = 5 
    for i, (query, expected_index) in enumerate(test_cases):
        top_k_indices = np.argsort(sims[i])[::-1][:K]
        
        if expected_index in top_k_indices:
            recalls.append(1.0)
            rank = np.where(top_k_indices == expected_index)[0][0] + 1
            aps.append(1.0 / rank)
            ndcgs.append(1.0 / math.log2(rank + 1))
        else:
            recalls.append(0.0)
            aps.append(0.0)
            ndcgs.append(0.0)
            
    avg_recall = np.mean(recalls)
    avg_map = np.mean(aps)
    avg_ndcg = np.mean(ndcgs)
    
    results_list.append({
        'Dimension': dim,
        'Memory (Bytes)': memory_bytes,
        'Latency (µs)': avg_latency_us,
        'Recall@5': avg_recall,
        'mAP@5': avg_map,
        'NDCG@5': avg_ndcg
    })

print("✅ Stress Test Completed!\n")

df_results = pd.DataFrame(results_list)
df_results.set_index('Dimension', inplace=True)

styled_df = (df_results.style
             .background_gradient(cmap='Greens_r', subset=['Memory (Bytes)', 'Latency (µs)'])
             .background_gradient(cmap='Greens', subset=['Recall@5', 'mAP@5', 'NDCG@5'])
             .format({
                 'Memory (Bytes)': '{:,.0f}',
                 'Latency (µs)': '{:.2f}',
                 'Recall@5': '{:.2f}',
                 'mAP@5': '{:.2f}',
                 'NDCG@5': '{:.2f}'
             }))

styled_df

⏳ Generating 1000 diverse and highly dense sentences for STRESS TEST...
⏳ Calling OpenAI/AvalAI API to generate 1000 embeddings...
   -> Fetching 1000 embeddings in chunks of 200...
      [+] Processed 200/1000
      [+] Processed 400/1000
      [+] Processed 600/1000
      [+] Processed 800/1000
      [+] Processed 1000/1000
   -> Fetching 4 embeddings in chunks of 10...
      [+] Processed 4/4

 Starting 1000-Item STRESS TEST Benchmark...
✅ Stress Test Completed!



,Memory (Bytes),Latency (µs),Recall@5,mAP@5,NDCG@5
Dimension,,,,,
1536,"12,288,000",7908.61,1.00,1.00,1.00
512,"4,096,000",2616.38,1.00,1.00,1.00
128,"1,024,000",525.40,1.00,1.00,1.00
64,"512,000",370.41,1.00,0.83,0.88
32,"256,000",307.90,1.00,0.52,0.64


---

### Ultimate Stress Test Dataset Generation (10000 times)

In [ ]:
def get_embeddings_in_chunks(texts: list, chunk_size=250) -> np.ndarray:
    """Fetch embeddings in safe chunks to avoid API Payload limits or Timeouts."""
    all_embeddings = []
    print(f"   -> Fetching {len(texts)} embeddings in chunks of {chunk_size} (This may take a few minutes)...")
    for i in range(0, len(texts), chunk_size):
        chunk = texts[i:i + chunk_size]
        try:
            response = client.embeddings.create(model=EMBEDDING_MODEL, input=chunk)
            sorted_data = sorted(response.data, key=lambda x: x.index)
            all_embeddings.extend([data.embedding for data in sorted_data])
            print(f"      [+] Processed {min(i + chunk_size, len(texts))}/{len(texts)} embeddings")
            time.sleep(0.5) # Anti-Rate-Limit delay
        except Exception as e:
            print(f"      [!] Error at chunk {i}: {e}. Retrying in 5 seconds...")
            time.sleep(5)
            # Retry one more time
            response = client.embeddings.create(model=EMBEDDING_MODEL, input=chunk)
            sorted_data = sorted(response.data, key=lambda x: x.index)
            all_embeddings.extend([data.embedding for data in sorted_data])
            
    return np.array(all_embeddings)

print("⏳ Generating 10,000 diverse and highly dense sentences for MASSIVE STRESS TEST...")
random.seed(42)

# Group A: 6000 Highly Similar Sentences (Tech Profiles)
# 20 Names * 15 Cities * 20 Skills = 6000 Combinations
names = ["Ali", "Sara", "Reza", "Mina", "Hassan", "Zahra", "Omid", "Neda", "Sina", "Maryam", 
         "Amir", "Mahsa", "Hadi", "Roya", "Farid", "Shirin", "Kian", "Tara", "Navid", "Leila"]
cities = ["Tehran", "Karaj", "Isfahan", "Shiraz", "Mashhad", "Tabriz", "Ahvaz", "Qom", 
          "Kermanshah", "Rasht", "Kerman", "Urmia", "Yazd", "Hamadan", "Ardabil"]
skills = ["Python", "Java", "C++", "JavaScript", "Go", "Rust", "Ruby", "Swift", "Kotlin", "PHP", 
          "TypeScript", "Scala", "Perl", "Haskell", "Lua", "Dart", "R", "MATLAB", "SQL", "Bash"]

tech_sentences = [f"The software engineer named {n} is currently residing in {c} and specializes in {s} development." 
                  for n, c, s in itertools.product(names, cities, skills)]

# Group B: 2000 Short Random Facts (Noise)
# 20 Subjects * 20 Actions * 5 Objects = 2000 Combinations
subjects = ["Water", "Fire", "Earth", "Air", "Space", "Time", "Gravity", "Light", "Sound", "Electricity", 
            "Magnetism", "Radiation", "Heat", "Cold", "Wind", "Rain", "Snow", "Ice", "Plasma", "Dust"]
actions = ["moves", "stops", "heats", "cools", "expands", "contracts", "falls", "rises", "bends", "breaks", 
           "forms", "dissolves", "reacts", "stabilizes", "accelerates", "slows", "reflects", "absorbs", "emits", "transfers"]
objects = ["quickly.", "slowly.", "naturally.", "artificially.", "spontaneously."]

short_sentences = [f"{sub} {act} {obj}" for sub, act, obj in itertools.product(subjects, actions, objects)]

# Group C: 2000 Long Complex Paragraphs (Semantic Distraction)
themes = ["Artificial Intelligence", "Climate Change", "Space Exploration", "Quantum Mechanics", "Ancient History"]
impacts = ["is transforming how we live and work daily,", "poses significant challenges to modern society,", "opens up entirely new frontiers of discovery,"]
details = ["requiring global cooperation and massive investment to ensure a sustainable future for all.", "often leading to unexpected philosophical and ethical debates among top scholars.", "pushing the boundaries of what humanity once thought was scientifically possible.", "which demands rigorous testing, validation, and peer review before implementation."]

long_sentences = []
for i in range(2000): # Create exactly 2000
    t = random.choice(themes)
    imp = random.choice(impacts)
    d = random.choice(details)
    long_sentences.append(f"The extensive study of {t} {imp} {d} This phenomenon has been heavily documented in numerous peer-reviewed research papers and academic journals (Reference Archive ID: {i + 1000}).")

# Combine all groups to create exactly 10,000 items
dataset_texts = tech_sentences + short_sentences + long_sentences
print(f"✅ Dataset successfully generated with {len(dataset_texts)} items.")

# Fetch Embeddings
print("\n⏳ Calling API to generate 10,000 embeddings...")
dataset_embeddings = get_embeddings_in_chunks(dataset_texts, chunk_size=250)

# Helper function to find target indices
def get_target_index(n, c, s):
    target = f"The software engineer named {n} is currently residing in {c} and specializes in {s} development."
    return dataset_texts.index(target)

# Define complex test cases
test_cases = [
    ("Looking for a Go developer based in Isfahan named Reza.", get_target_index("Reza", "Isfahan", "Go")),
    ("I need a Rust programmer from Mashhad, ideally her name is Mina.", get_target_index("Mina", "Mashhad", "Rust")),
    ("Where does Hassan live and what does he code? He writes Java in Shiraz.", get_target_index("Hassan", "Shiraz", "Java")),
    ("Find the profile for Ali. He is a C++ expert located in Tehran.", get_target_index("Ali", "Tehran", "C++")),
    ("Do we have a Swift developer residing in Yazd named Navid?", get_target_index("Navid", "Yazd", "Swift"))
]

queries = [tc[0] for tc in test_cases]
query_embeddings_full = get_embeddings_in_chunks(queries, chunk_size=10)

# Ultimate Matryoshka Benchmarking
dimensions_to_test = [1536, 512, 128, 64, 32] 
results_list = []

print("\n🚀 Starting 10,000-Item STRESS TEST Benchmark...")

for dim in dimensions_to_test:
    trunc_dataset = dataset_embeddings[:, :dim]
    trunc_queries = query_embeddings_full[:, :dim]
    
    trunc_dataset = trunc_dataset / np.linalg.norm(trunc_dataset, axis=1, keepdims=True)
    trunc_queries = trunc_queries / np.linalg.norm(trunc_queries, axis=1, keepdims=True)
    
    memory_bytes = trunc_dataset.nbytes
    
    # 50 iterations for stable latency testing (10000x matrix is larger, so 50 is enough to get a good average)
    start_time = time.perf_counter()
    for _ in range(50): 
        _ = cosine_similarity(trunc_queries, trunc_dataset)
    end_time = time.perf_counter()
    
    avg_latency_us = ((end_time - start_time) / 50) * 1_000_000 
    
    recalls = []
    aps = []
    ndcgs = []
    
    sims = cosine_similarity(trunc_queries, trunc_dataset)
    
    # Strict Recall@5
    K = 5 
    for i, (query, expected_index) in enumerate(test_cases):
        top_k_indices = np.argsort(sims[i])[::-1][:K]
        
        if expected_index in top_k_indices:
            recalls.append(1.0)
            rank = np.where(top_k_indices == expected_index)[0][0] + 1
            aps.append(1.0 / rank)
            ndcgs.append(1.0 / math.log2(rank + 1))
        else:
            recalls.append(0.0)
            aps.append(0.0)
            ndcgs.append(0.0)
            
    avg_recall = np.mean(recalls)
    avg_map = np.mean(aps)
    avg_ndcg = np.mean(ndcgs)
    
    results_list.append({
        'Dimension': dim,
        'Memory (Bytes)': memory_bytes,
        'Latency (µs)': avg_latency_us,
        'Recall@5': avg_recall,
        'mAP@5': avg_map,
        'NDCG@5': avg_ndcg
    })

print("✅ Massive Stress Test Completed!\n")

# Display Results
df_results = pd.DataFrame(results_list)
df_results.set_index('Dimension', inplace=True)

styled_df = (df_results.style
             .background_gradient(cmap='Greens_r', subset=['Memory (Bytes)', 'Latency (µs)'])
             .background_gradient(cmap='Greens', subset=['Recall@5', 'mAP@5', 'NDCG@5'])
             .format({
                 'Memory (Bytes)': '{:,.0f}',
                 'Latency (µs)': '{:.2f}',
                 'Recall@5': '{:.2f}',
                 'mAP@5': '{:.2f}',
                 'NDCG@5': '{:.2f}'
             }))

styled_df

⏳ Generating 10,000 diverse and highly dense sentences for MASSIVE STRESS TEST...
✅ Dataset successfully generated with 10000 items.

⏳ Calling API to generate 10,000 embeddings...
   -> Fetching 10000 embeddings in chunks of 250 (This may take a few minutes)...
      [+] Processed 250/10000 embeddings
      [+] Processed 500/10000 embeddings
      [+] Processed 750/10000 embeddings
      [+] Processed 1000/10000 embeddings
      [+] Processed 1250/10000 embeddings
      [+] Processed 1500/10000 embeddings
      [+] Processed 1750/10000 embeddings
      [+] Processed 2000/10000 embeddings
      [+] Processed 2250/10000 embeddings
      [+] Processed 2500/10000 embeddings
      [+] Processed 2750/10000 embeddings
      [+] Processed 3000/10000 embeddings
      [+] Processed 3250/10000 embeddings
      [+] Processed 3500/10000 embeddings
      [+] Processed 3750/10000 embeddings
      [+] Processed 4000/10000 embeddings
      [+] Processed 4250/10000 embeddings
      [+] Processed 4500/1

,Memory (Bytes),Latency (µs),Recall@5,mAP@5,NDCG@5
Dimension,,,,,
1536,"122,880,000",82448.63,1.00,1.00,1.00
512,"40,960,000",24168.22,1.00,1.00,1.00
128,"10,240,000",6056.49,1.00,1.00,1.00
64,"5,120,000",2823.44,1.00,0.85,0.89
32,"2,560,000",1422.22,0.60,0.28,0.35


---

## ***Stress Test: Deep Analysis & Insights***

To rigorously evaluate the limits of Matryoshka Representation Learning (MRL), we scaled the dataset to **10,000 highly dense vectors** (including 6,000 extremely similar profiles and 4,000 noise/distraction texts). The results expose the true behavior of dimensional truncation in a simulated production environment.

#### 1. Infrastructure Scaling (Memory & Latency)
The benchmark perfectly demonstrates linear resource scaling:
* **Memory Cost:** The full 1536-dimensional matrix consumes ~122.8 MB of RAM. Truncating to 128 dimensions reduces this to ~10.2 MB (a 91.6% reduction).
* **Search Latency:** The compute time dropped from ~82.4k µs to just ~1.4k µs at 32 dimensions, proving that lower dimensions drastically reduce the computational overhead of dot-product operations.

#### 2. The Matryoshka "Sweet Spot" (128 Dimensions)
The most remarkable finding is the performance at **128 dimensions**. Despite stripping away 91.6% of the original vector data, the model maintained a flawless score of **1.00 across all metrics** (Recall@5, mAP@5, NDCG@5). This proves that the `text-embedding-3-small` model successfully front-loaded all distinguishing semantic variables for 10,000 distinct items into just the first 128 floats. For a production system, choosing 128 dimensions offers maximum cost efficiency with zero accuracy penalty.

#### 3. The Breaking Point & Vector Collision (64 and 32 Dimensions)
As we pushed the truncation further, the physical limits of the mathematical space were exposed:
* **At 64 Dimensions (Rank Degradation):** The `Recall@5` remained 1.00 (the correct answer was still retrieved in the top 5). However, `mAP@5` dropped to 0.85 and `NDCG@5` to 0.89. This indicates that while the system found the target, it could no longer confidently place it at **Rank 1**. The high density of 6,000 similar engineers caused slight confusion in ranking.
* **At 32 Dimensions (Structural Collapse):** At this extreme compression, the vector space ran out of capacity to differentiate subtle nuances (e.g., distinguishing "Java in Shiraz" from "Python in Shiraz"). `Recall@5` plummeted to **0.60**, meaning 40% of the time, the correct answer was entirely pushed out of the top 5 results. The `mAP@5` dropping to 0.28 signifies severe **Vector Collision**, where non-relevant vectors mathematically overlapped with the target vectors.

#### Conclusion
This stress test empirically proves that while Matryoshka embeddings are revolutionary for cost-saving, aggressive truncation (below 128 dimensions) in highly dense, semantically overlapping datasets leads to fatal information loss. The architecture must carefully balance the trade-off between vector size and dataset complexity.